# CLI vs API: Two Doors Into the Same House
<img src='../../images/xebia-logo.png' width='300px' align='right' style="padding: 15px">

Both a **CLI** (Command Line Interface) and an **API** (Application Programming Interface) are *interfaces*: ways to interact with your code. The business logic stays the same; only the **door** changes.

## Before you start Day 3

Run these two commands once, in a terminal in the root folder of the repository. They add the packages that today's notebooks use.

```bash
uv add fastapi uvicorn requests pydantic-settings typer
uv add --dev httpx2
```

- **fastapi**: the framework for building APIs
- **uvicorn**: the server that runs a FastAPI app
- **requests**: to call an API from Python
- **pydantic-settings**: settings from environment variables, at the end of the Pydantic notebook
- **typer**: the library for command-line tools
- **httpx2**: needed by FastAPI's `TestClient` to test an API without starting a server; it is only a development tool, hence `--dev`

## What Is an Interface?

Your code does something useful: it trains a model, processes data or calculates prices. But code sitting in a `.py` file doesn't do anything until someone *calls* it.

An **interface** is the layer that lets someone (a human or another program) trigger your code:

```
┌──────────────────────────────────────────────────┐
│                                                  │
│   Human at terminal ──► CLI ──► Business Logic   │
│                                                  │
│   Web app / Service ──► API ──► Business Logic   │
│                                                  │
│   Notebook cell ─────► import ► Business Logic   │
│                                                  │
└──────────────────────────────────────────────────┘
```

The **same business logic** can be exposed through multiple interfaces simultaneously.

## What Is a CLI?

A **Command Line Interface** lets a human interact with your code by typing commands in a terminal.

```bash
python train.py --data train.csv --epochs 10 --learning-rate 0.001
```

### Characteristics

| Property | Description |
|----------|-------------|
| **User** | A human typing in a terminal (or a script/cron job) |
| **Communication** | Arguments, flags, and options passed at invocation |
| **Lifecycle** | One-shot: runs, prints output, exits |
| **Output** | Text printed to stdout/stderr |
| **State** | Stateless: each run starts fresh |
| **Concurrency** | One user, one execution at a time |
| **Discovery** | `--help` flag shows available commands and options |

### Real-World CLI Examples

You already use CLIs every day:

| Tool | Example Command | What It Does |
|------|----------------|---------------|
| **git** | `git commit -m "fix bug"` | Version control |
| **docker** | `docker build -t myapp .` | Container management |
| **uv** | `uv sync` | Python package management |
| **pytest** | `pytest tests/ -v` | Run tests |
| **ruff** | `ruff check src/` | Lint Python code |
| **kubectl** | `kubectl get pods` | Manage Kubernetes clusters |

### When to Build a CLI

- **Data pipelines**: `python etl.py --source s3://bucket --date 2026-04-30`
- **Model training**: `python train.py --config config.yaml`
- **Database migrations**: `python migrate.py --up`
- **DevOps automation**: scripts run by CI/CD, cron jobs, or humans
- **Developer tools**: linters, formatters, code generators

## What Is an API?

An **Application Programming Interface** (specifically a web/REST API) lets *other programs* interact with your code over the network via HTTP.

```bash
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"features": [1.2, 3.4, 5.6]}'
```

### Characteristics

| Property | Description |
|----------|-------------|
| **User** | Other programs, frontend apps, mobile apps, browsers |
| **Communication** | HTTP requests (JSON bodies, headers, query params) |
| **Lifecycle** | Long-running server: starts, listens, handles many requests |
| **Output** | Structured data (JSON, XML) with status codes |
| **State** | Can be stateful (database, sessions) or stateless |
| **Concurrency** | Handles many users/requests simultaneously |
| **Discovery** | Swagger/OpenAPI docs at `/docs` (FastAPI generates this automatically) |

### Real-World API Examples

| Service | What It Exposes | Who Calls It |
|---------|----------------|---------------|
| **Stripe** | Payment processing | E-commerce websites |
| **OpenAI** | LLM inference | Chat apps, developer tools |
| **Google Maps** | Geocoding, directions | Mobile apps, websites |
| **Spotify** | Music catalog, playback | Mobile/desktop clients |
| **Your ML model** | Predictions | Frontend dashboards, other microservices |

### When to Build an API

- **Serving ML models**: A frontend sends features, your API returns predictions
- **Microservices**: Service A calls Service B's API to get data
- **Mobile/web apps**: The backend that powers a user-facing application
- **Data access**: Exposing a database or data lake to authorized consumers
- **Integration**: Letting third-party tools interact with your system (webhooks, plugins)

## Side-by-Side Comparison

| | **CLI** | **API** |
|---|---|---|
| **Who uses it** | Humans in a terminal, scripts, CI/CD | Other programs, frontends, services |
| **How it communicates** | Command-line arguments → printed text | HTTP requests (JSON) → HTTP responses (JSON) |
| **Runs as** | One-shot command (run → exit) | Long-running server (start → listen → handle) |
| **Invocation** | `python train.py --epochs 10` | `POST /train {"epochs": 10}` |
| **Concurrency** | One execution at a time | Many simultaneous requests |
| **Where it runs** | Locally, on the same machine | Anywhere reachable over a network |
| **Input validation** | argparse / Typer type hints | Pydantic models (automatic in FastAPI) |
| **Documentation** | `--help` flag | Swagger UI at `/docs` |
| **Testing** | `subprocess` or Typer's `CliRunner` | `TestClient` (no server needed) |
| **Python tools** | argparse, Click, **Typer** | Flask, Django, **FastAPI** |

## The Golden Rule: Separate Logic from Interface

The most important pattern, whether you're building a CLI or an API, is:

> **Your business logic should NOT know how it's being called.**

The logic should be in a **package** (like `animal_shelter`). The interfaces should be **thin wrappers** that import and call it.

In [1]:
# === Business Logic (in your package, e.g. animal_shelter/predict.py) ===

def predict(features: list[float]) -> float:
    """Pure business logic: no CLI, no HTTP, no side effects."""
    # In reality this would load a model and predict
    return sum(features) / len(features)

In [2]:
# === CLI wrapper (thin) ===
# In a real project this would be in packages/cli/src/cli/cli.py

import typer

app = typer.Typer()

@app.command()
def cli_predict(features: list[float]):
    """Run prediction from the terminal."""
    result = predict(features)  # calls the SAME logic
    typer.echo(f"Prediction: {result}")

In [3]:
# === API wrapper (thin) ===
# In a real project this would be in packages/api/src/api/app.py

from fastapi import FastAPI
from pydantic import BaseModel

api = FastAPI()

class PredictRequest(BaseModel):
    features: list[float]

class PredictResponse(BaseModel):
    prediction: float

@api.post("/predict")
async def api_predict(req: PredictRequest) -> PredictResponse:
    """Run prediction over HTTP."""
    result = predict(req.features)  # calls the SAME logic
    return PredictResponse(prediction=result)

Notice how both wrappers are just **3-5 lines** that call `predict()`. All the real work is in the business logic, which:

- Has **no idea** whether it's being called from a terminal or a web request
- Can be **unit tested** directly without any interface
- Can be **reused** by adding new interfaces (e.g., a Streamlit dashboard) without changing a single line

## Project Structure: Putting It All Together

In a production project, this separation is reflected in the folder structure:

```
my-project/
├── src/
│   └── animal_shelter/      ← Business logic (models, data, features)
│       ├── __init__.py
│       ├── data.py
│       ├── features.py
│       └── predict.py
├── packages/
│   ├── cli/                 ← CLI interface (Typer)
│   │   ├── pyproject.toml
│   │   └── src/cli/cli.py
│   └── api/                 ← API interface (FastAPI)
│       ├── pyproject.toml
│       └── src/api/app.py
├── tests/                   ← Tests for the business logic
├── pyproject.toml           ← Main project config
└── ...
```

Each interface is a **uv workspace member**: an independent package with its own dependencies, but they all share the same virtual environment and can import `animal_shelter`.

## Can You Use Both at the Same Time?

Absolutely! In fact, this is very common in production:

| Scenario | CLI | API |
|----------|-----|-----|
| **ML workflow** | `cli train --data train.csv` (batch training on a schedule) | `POST /predict` (real-time serving) |
| **Data pipeline** | `cli ingest --source s3://...` (triggered by Airflow) | `GET /status` (monitoring dashboard) |
| **Admin tasks** | `cli create-user --name Alice` (ops team) | `POST /users` (frontend signup) |

The CLI handles **batch/offline/admin** tasks and the API handles **online/real-time/multi-user** tasks. Both call the same tested business logic underneath.

## Key Takeaways

1. **CLI** = terminal door, **API** = network door: both lead to the same logic
2. **Separate business logic from interfaces**: this is the single most important pattern
3. Use **Typer** for CLIs and **FastAPI** for APIs: both are modern, type-hint-driven, and easy to test
4. In a real project, interfaces live in their own packages (`packages/cli`, `packages/api`) as **uv workspace members**
5. You can (and should) have **both** a CLI and an API for different use cases